# NASDAQ Opening Gap Forecast: model development

Predicting whether the NASDAQ-100 opens up, down or flat from the overnight moves of seven Asian and European stock indices. The data is not included: see `data/README.md`.

## Step 1: Problem statement

Our project helps the client by generating a daily prediction of the expected opening gap in the NASDAQ-100, using overnight movements from major global stock indices such as the Nikkei-225, DAX-40, and FTSE-100. This predictive signal enables the client to be better informed and strategically prepared for potential market volatility at the opening bell. By anticipating whether the market will gap up or down, the client can support faster, more confident pre-market decisions, improve client hedge positioning, and strengthen the value of their real-time analytics offerings. This enhances their competitive edge and positions them as a forward-thinking leader in data-driven trading insights.

## Step 2: Load and explore data

In the project the daily index prices were stored in a PostgreSQL database that the team designed, and loaded from there. In this repository the same data is read from the exported CSV files in the cells below.

## Step 3: Preprocessing

In [5]:
# Import libraries
import pandas as pd
import numpy as np

#### Loading and localizing the data

In [6]:
# CAC Data load
CAC_data = pd.read_csv('Data/CAC_40.csv', index_col="Date", parse_dates=True)
print(CAC_data.index[:5])
CAC_data.index = CAC_data.index.tz_localize("UTC")

# DAX Data load
DAX_data = pd.read_csv("Data/DAX_40.csv", index_col="Date", parse_dates=True)
DAX_data.index = DAX_data.index.tz_localize("UTC")

# FTSE Data load
FTSE_data = pd.read_csv("Data/FTSE_100.csv", index_col="Date", parse_dates=True)
FTSE_data.index = FTSE_data.index.tz_localize("UTC")

# Hang_Seng Data load
Hang_Seng_data = pd.read_csv("Data/Hang_Seng.csv", index_col="Date", parse_dates=True)
Hang_Seng_data.index = Hang_Seng_data.index.tz_localize("UTC")

# NIFTY Data load
NIFTY_data = pd.read_csv("Data/NIFTY_50.csv", index_col="Date", parse_dates=True)
NIFTY_data.index = NIFTY_data.index.tz_localize("UTC")

# Nikkei Data load
Nikkei_data = pd.read_csv("Data/Nikkei_225.csv", index_col="Date", parse_dates=True)
Nikkei_data.index = Nikkei_data.index.tz_localize("UTC")

# Shanghai_Composite Data load
Shanghai_Composite_data = pd.read_csv("Data/Shanghai_Composite.csv", index_col="Date", parse_dates=True)
Shanghai_Composite_data.index = Shanghai_Composite_data.index.tz_localize("UTC")


DatetimeIndex(['2015-02-12', '2015-02-13', '2015-02-16', '2015-02-17',
               '2015-02-18'],
              dtype='datetime64[ns]', name='Date', freq=None)


In [7]:
# Drop the 'High', 'Low' and 'Volume' columns from CAC data
CAC_data = CAC_data.drop(columns=['High', 'Low', 'Volume'])

# Drop the 'High', 'Low' and 'Volume' columns from DAX data
DAX_data = DAX_data.drop(columns=['High', 'Low', 'Volume'])

# Drop the 'High', 'Low' and 'Volume' columns from FTSE data
FTSE_data = FTSE_data.drop(columns=['High', 'Low', 'Volume'])

# Drop the 'High', 'Low' and 'Volume' columns from Hang_Seng data
Hang_Seng_data = Hang_Seng_data.drop(columns=['High', 'Low', 'Volume'])

# Drop the 'High', 'Low' and 'Volume' columns from NIFTY data
NIFTY_data = NIFTY_data.drop(columns=['High', 'Low', 'Volume'])

# Drop the 'High', 'Low' and 'Volume' columns from Nikkei data
Nikkei_data = Nikkei_data.drop(columns=['High', 'Low', 'Volume'])

# Drop the 'High', 'Low' and 'Volume' columns from Shanghai_Composite data
Shanghai_Composite_data = Shanghai_Composite_data.drop(columns=['High', 'Low', 'Volume'])

In [8]:
CAC_data.head()

,Open,Close,Adj Close
Date,,,
2015-02-12 00:00:00+00:00,4670.080078,4726.200195,4726.200195
2015-02-13 00:00:00+00:00,4748.040039,4759.359863,4759.359863
2015-02-16 00:00:00+00:00,4757.520020,4751.950195,4751.950195
2015-02-17 00:00:00+00:00,4720.930176,4753.990234,4753.990234
2015-02-18 00:00:00+00:00,4782.589844,4799.029785,4799.029785


## Missing Values

### Checking for Missing Values for CAC 40

In [9]:
# Check for NaN values in the dataset
missing_values = CAC_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open         0
Close        0
Adj Close    0
dtype: int64


In [10]:
# Find the initial date of the dataset
CAC_start = CAC_data.index.min()

print(CAC_start)

2015-02-12 00:00:00+00:00


In [11]:
# Find the end date of the dataset
CAC_end = CAC_data.index.max()

print(CAC_end)

2025-02-11 00:00:00+00:00


In [12]:
# Generate a complete range of business days
CAC_all_workdays = pd.date_range(start=CAC_start, end=CAC_end, freq='B')

print(CAC_all_workdays)

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-29 00:00:00+00:00', '2025-01-30 00:00:00+00:00',
               '2025-01-31 00:00:00+00:00', '2025-02-03 00:00:00+00:00',
               '2025-02-04 00:00:00+00:00', '2025-02-05 00:00:00+00:00',
               '2025-02-06 00:00:00+00:00', '2025-02-07 00:00:00+00:00',
               '2025-02-10 00:00:00+00:00', '2025-02-11 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2609, freq='B')


In [13]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'CAC_day' index
CAC_missing_dates = CAC_all_workdays.difference(CAC_data.index)

print("Missing workdays:\n", CAC_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-04-03 00:00:00+00:00', '2015-04-06 00:00:00+00:00',
               '2015-05-01 00:00:00+00:00', '2015-12-25 00:00:00+00:00',
               '2015-12-31 00:00:00+00:00', '2016-01-01 00:00:00+00:00',
               '2016-03-25 00:00:00+00:00', '2016-03-28 00:00:00+00:00',
               '2016-12-26 00:00:00+00:00', '2017-04-14 00:00:00+00:00',
               '2017-04-17 00:00:00+00:00', '2017-05-01 00:00:00+00:00',
               '2017-12-25 00:00:00+00:00', '2017-12-26 00:00:00+00:00',
               '2018-01-01 00:00:00+00:00', '2018-03-30 00:00:00+00:00',
               '2018-04-02 00:00:00+00:00', '2018-05-01 00:00:00+00:00',
               '2018-12-25 00:00:00+00:00', '2018-12-26 00:00:00+00:00',
               '2019-01-01 00:00:00+00:00', '2019-04-19 00:00:00+00:00',
               '2019-04-22 00:00:00+00:00', '2019-05-01 00:00:00+00:00',
               '2019-12-25 00:00:00+00:00', '2019-12-26 00:00:00+00:00',
               '2020-01-01 00:00

In [14]:
# Print the number of missing workdays by getting the length of the 'missing_dates' array
print("Number of missing workdays over the entire time range:", CAC_missing_dates.shape[0])

Number of missing workdays over the entire time range: 49


### Checking for Missing Values for DAX 40


In [15]:
# Check for NaN values in the dataset
missing_values = DAX_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open         0
Close        0
Adj Close    0
dtype: int64


In [16]:
# Find the initial date of the dataset
DAX_start = DAX_data.index.min()

print(DAX_start)

2015-02-12 00:00:00+00:00


In [17]:
# Find the end date of the dataset
DAX_end = DAX_data.index.max()

print(DAX_end)

2025-02-11 00:00:00+00:00


In [18]:
# Generate a complete range of business days
DAX_all_workdays = pd.date_range(start=DAX_start, end=DAX_end, freq='B')
DAX_all_workdays

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-29 00:00:00+00:00', '2025-01-30 00:00:00+00:00',
               '2025-01-31 00:00:00+00:00', '2025-02-03 00:00:00+00:00',
               '2025-02-04 00:00:00+00:00', '2025-02-05 00:00:00+00:00',
               '2025-02-06 00:00:00+00:00', '2025-02-07 00:00:00+00:00',
               '2025-02-10 00:00:00+00:00', '2025-02-11 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2609, freq='B')

In [19]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'DAX_data' index
DAX_missing_dates = DAX_all_workdays.difference(DAX_data.index)

print("Missing workdays:\n", DAX_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-04-03 00:00:00+00:00', '2015-04-06 00:00:00+00:00',
               '2015-05-01 00:00:00+00:00', '2015-05-25 00:00:00+00:00',
               '2015-12-24 00:00:00+00:00', '2015-12-25 00:00:00+00:00',
               '2015-12-31 00:00:00+00:00', '2016-01-01 00:00:00+00:00',
               '2016-03-25 00:00:00+00:00', '2016-03-28 00:00:00+00:00',
               '2016-05-16 00:00:00+00:00', '2016-10-03 00:00:00+00:00',
               '2016-12-26 00:00:00+00:00', '2017-04-14 00:00:00+00:00',
               '2017-04-17 00:00:00+00:00', '2017-05-01 00:00:00+00:00',
               '2017-06-05 00:00:00+00:00', '2017-10-03 00:00:00+00:00',
               '2017-10-31 00:00:00+00:00', '2017-12-25 00:00:00+00:00',
               '2017-12-26 00:00:00+00:00', '2018-01-01 00:00:00+00:00',
               '2018-03-30 00:00:00+00:00', '2018-04-02 00:00:00+00:00',
               '2018-05-01 00:00:00+00:00', '2018-05-21 00:00:00+00:00',
               '2018-10-03 00:00

In [20]:
# Print the number of missing workdays by getting the length of the 'missing_dates' array
print("Number of missing workdays over the entire time range:", DAX_missing_dates.shape[0])

Number of missing workdays over the entire time range: 72


### Checking for Missing Values for FTSE 100


In [21]:
# Check for NaN values in the dataset
missing_values = FTSE_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open         0
Close        0
Adj Close    0
dtype: int64


In [22]:
# Find the initial date of the dataset
FTSE_start = FTSE_data.index.min()
FTSE_start

Timestamp('2015-02-12 00:00:00+0000', tz='UTC')

In [23]:
# Find the end date of the dataset
FTSE_end = FTSE_data.index.max()
FTSE_end

Timestamp('2025-02-11 00:00:00+0000', tz='UTC')

In [24]:
# Generate a complete range of business days
FTSE_all_workdays = pd.date_range(start=FTSE_start, end=FTSE_end, freq='B')
FTSE_all_workdays

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-29 00:00:00+00:00', '2025-01-30 00:00:00+00:00',
               '2025-01-31 00:00:00+00:00', '2025-02-03 00:00:00+00:00',
               '2025-02-04 00:00:00+00:00', '2025-02-05 00:00:00+00:00',
               '2025-02-06 00:00:00+00:00', '2025-02-07 00:00:00+00:00',
               '2025-02-10 00:00:00+00:00', '2025-02-11 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2609, freq='B')

In [25]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'DAX_data' index
FTSE_missing_dates = FTSE_all_workdays.difference(FTSE_data.index)

print("Missing workdays:\n", FTSE_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-04-03 00:00:00+00:00', '2015-04-06 00:00:00+00:00',
               '2015-05-04 00:00:00+00:00', '2015-05-25 00:00:00+00:00',
               '2015-08-31 00:00:00+00:00', '2015-12-25 00:00:00+00:00',
               '2015-12-28 00:00:00+00:00', '2016-01-01 00:00:00+00:00',
               '2016-03-25 00:00:00+00:00', '2016-03-28 00:00:00+00:00',
               '2016-05-02 00:00:00+00:00', '2016-05-30 00:00:00+00:00',
               '2016-08-29 00:00:00+00:00', '2016-12-26 00:00:00+00:00',
               '2016-12-27 00:00:00+00:00', '2017-01-02 00:00:00+00:00',
               '2017-04-14 00:00:00+00:00', '2017-04-17 00:00:00+00:00',
               '2017-05-01 00:00:00+00:00', '2017-05-29 00:00:00+00:00',
               '2017-08-28 00:00:00+00:00', '2017-12-25 00:00:00+00:00',
               '2017-12-26 00:00:00+00:00', '2018-01-01 00:00:00+00:00',
               '2018-03-30 00:00:00+00:00', '2018-04-02 00:00:00+00:00',
               '2018-05-07 00:00

In [26]:
print("Number of missing workdays over the entire time range:", FTSE_missing_dates.shape[0])

Number of missing workdays over the entire time range: 84


In [27]:
# 1 missing workday in the FTSE 100 dataset (2020-12-22)

Given that there was only one date identified as missing from the dataset, it can be ignored for the purposes of analysis.


### Checking for Missing Values for Hang Seng


In [28]:
# Check for NaN values in the dataset
missing_values = Hang_Seng_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open         0
Close        0
Adj Close    0
dtype: int64


In [29]:
# Find the initial date of the dataset
Hang_Seng_start = Hang_Seng_data.index.min()
print(Hang_Seng_start)

2015-02-12 00:00:00+00:00


In [30]:
# Find the end date of the dataset
Hang_Seng_end = Hang_Seng_data.index.max()
print(Hang_Seng_end)

2025-02-11 00:00:00+00:00


In [31]:
# Generate a complete range of business days
Hang_Seng_all_workdays = pd.date_range(start=Hang_Seng_start, end=Hang_Seng_end, freq='B')
print(Hang_Seng_all_workdays)

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-29 00:00:00+00:00', '2025-01-30 00:00:00+00:00',
               '2025-01-31 00:00:00+00:00', '2025-02-03 00:00:00+00:00',
               '2025-02-04 00:00:00+00:00', '2025-02-05 00:00:00+00:00',
               '2025-02-06 00:00:00+00:00', '2025-02-07 00:00:00+00:00',
               '2025-02-10 00:00:00+00:00', '2025-02-11 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2609, freq='B')


In [32]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'Hang_Seng_data' index
Hang_Seng_missing_dates = Hang_Seng_all_workdays.difference(Hang_Seng_data.index)
print("Missing workdays:\n", Hang_Seng_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-02-19 00:00:00+00:00', '2015-02-20 00:00:00+00:00',
               '2015-04-03 00:00:00+00:00', '2015-04-06 00:00:00+00:00',
               '2015-04-07 00:00:00+00:00', '2015-05-01 00:00:00+00:00',
               '2015-05-25 00:00:00+00:00', '2015-07-01 00:00:00+00:00',
               '2015-09-03 00:00:00+00:00', '2015-09-28 00:00:00+00:00',
               ...
               '2024-09-06 00:00:00+00:00', '2024-09-18 00:00:00+00:00',
               '2024-10-01 00:00:00+00:00', '2024-10-11 00:00:00+00:00',
               '2024-12-25 00:00:00+00:00', '2024-12-26 00:00:00+00:00',
               '2025-01-01 00:00:00+00:00', '2025-01-29 00:00:00+00:00',
               '2025-01-30 00:00:00+00:00', '2025-01-31 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=152, freq=None)


In [33]:
# Print the number of missing workdays by getting the length of the 'missing_dates' array
print("Number of missing workdays over the entire time range:", Hang_Seng_missing_dates.shape[0])

Number of missing workdays over the entire time range: 152


### Checking for Missing Values for NIFTY 50

In [34]:
# Check for NaN values in the dataset
missing_values = NIFTY_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open         0
Close        0
Adj Close    0
dtype: int64


In [35]:
# Find the initial date of the dataset
NIFTY_start = NIFTY_data.index.min()
print(NIFTY_start)

2015-02-12 00:00:00+00:00


In [36]:
# Find the end date of the dataset
NIFTY_end = NIFTY_data.index.max()
print(NIFTY_end)

2025-02-11 00:00:00+00:00


In [37]:
# Generate a complete range of business days
NIFTY_all_workdays = pd.date_range(start=NIFTY_start, end=NIFTY_end, freq='B')
print(NIFTY_all_workdays)

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-29 00:00:00+00:00', '2025-01-30 00:00:00+00:00',
               '2025-01-31 00:00:00+00:00', '2025-02-03 00:00:00+00:00',
               '2025-02-04 00:00:00+00:00', '2025-02-05 00:00:00+00:00',
               '2025-02-06 00:00:00+00:00', '2025-02-07 00:00:00+00:00',
               '2025-02-10 00:00:00+00:00', '2025-02-11 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2609, freq='B')


In [38]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'NIFTY_data' index
NIFTY_missing_dates = NIFTY_all_workdays.difference(NIFTY_data.index)
print("Missing workdays:\n", NIFTY_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-02-17 00:00:00+00:00', '2015-03-06 00:00:00+00:00',
               '2015-04-02 00:00:00+00:00', '2015-04-03 00:00:00+00:00',
               '2015-04-14 00:00:00+00:00', '2015-04-15 00:00:00+00:00',
               '2015-05-01 00:00:00+00:00', '2015-09-17 00:00:00+00:00',
               '2015-09-25 00:00:00+00:00', '2015-10-02 00:00:00+00:00',
               ...
               '2024-04-17 00:00:00+00:00', '2024-05-01 00:00:00+00:00',
               '2024-05-20 00:00:00+00:00', '2024-06-17 00:00:00+00:00',
               '2024-07-17 00:00:00+00:00', '2024-08-15 00:00:00+00:00',
               '2024-10-02 00:00:00+00:00', '2024-11-15 00:00:00+00:00',
               '2024-11-20 00:00:00+00:00', '2024-12-25 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=148, freq=None)


In [39]:
# Print the number of missing workdays by getting the length of the 'missing_dates' array
print("Number of missing workdays over the entire time range:", NIFTY_missing_dates.shape[0])

Number of missing workdays over the entire time range: 148


In [40]:
# 5 missing workdays in the NIFTY 50 dataset (2024-05-20, 2024-06-17, 2024-07-17, 2024-11-15, 2024-11-20)

### Checking for Missing Values for Nikkei 225

In [41]:
# Check for NaN values in the dataset
missing_values = Nikkei_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open         0
Close        0
Adj Close    0
dtype: int64


In [42]:
# Find the initial date of the dataset
Nikkei_start = Nikkei_data.index.min()
print(Nikkei_start)

2015-02-12 00:00:00+00:00


In [43]:
# Find the end date of the dataset
Nikkei_end = Nikkei_data.index.max()
print(Nikkei_end)

2025-02-10 00:00:00+00:00


In [44]:

# Generate a complete range of business days
Nikkei_all_workdays = pd.date_range(start=Nikkei_start, end=Nikkei_end, freq='B')
print(Nikkei_all_workdays)

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-28 00:00:00+00:00', '2025-01-29 00:00:00+00:00',
               '2025-01-30 00:00:00+00:00', '2025-01-31 00:00:00+00:00',
               '2025-02-03 00:00:00+00:00', '2025-02-04 00:00:00+00:00',
               '2025-02-05 00:00:00+00:00', '2025-02-06 00:00:00+00:00',
               '2025-02-07 00:00:00+00:00', '2025-02-10 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2608, freq='B')


In [45]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'Nikkei_data' index
Nikkei_missing_dates = Nikkei_all_workdays.difference(Nikkei_data.index)
print("Missing workdays:\n", Nikkei_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-04-29 00:00:00+00:00', '2015-05-04 00:00:00+00:00',
               '2015-05-05 00:00:00+00:00', '2015-05-06 00:00:00+00:00',
               '2015-07-20 00:00:00+00:00', '2015-09-21 00:00:00+00:00',
               '2015-09-22 00:00:00+00:00', '2015-09-23 00:00:00+00:00',
               '2015-10-12 00:00:00+00:00', '2015-11-03 00:00:00+00:00',
               ...
               '2024-08-12 00:00:00+00:00', '2024-09-16 00:00:00+00:00',
               '2024-09-23 00:00:00+00:00', '2024-10-14 00:00:00+00:00',
               '2024-11-04 00:00:00+00:00', '2024-12-31 00:00:00+00:00',
               '2025-01-01 00:00:00+00:00', '2025-01-02 00:00:00+00:00',
               '2025-01-03 00:00:00+00:00', '2025-01-13 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=164, freq=None)


In [46]:
# Print the number of missing workdays by getting the length of the 'missing_dates' array
print("Number of missing workdays over the entire time range:", Nikkei_missing_dates.shape[0])

Number of missing workdays over the entire time range: 164


### Checking for Missing Values for Shanghai Composite

In [47]:
# Find the initial date of the dataset
Shanghai_Composite_start = Shanghai_Composite_data.index.min()
print(Shanghai_Composite_start)

2015-02-12 00:00:00+00:00


In [48]:
# Find the end date of the dataset
Shanghai_Composite_end = Shanghai_Composite_data.index.max()
print(Shanghai_Composite_end)

2025-02-11 00:00:00+00:00


In [49]:
# Generate a complete range of business days
Shanghai_Composite_all_workdays = pd.date_range(start=Shanghai_Composite_start, end=Shanghai_Composite_end, freq='B')
print(Shanghai_Composite_all_workdays)

DatetimeIndex(['2015-02-12 00:00:00+00:00', '2015-02-13 00:00:00+00:00',
               '2015-02-16 00:00:00+00:00', '2015-02-17 00:00:00+00:00',
               '2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-02-25 00:00:00+00:00',
               ...
               '2025-01-29 00:00:00+00:00', '2025-01-30 00:00:00+00:00',
               '2025-01-31 00:00:00+00:00', '2025-02-03 00:00:00+00:00',
               '2025-02-04 00:00:00+00:00', '2025-02-05 00:00:00+00:00',
               '2025-02-06 00:00:00+00:00', '2025-02-07 00:00:00+00:00',
               '2025-02-10 00:00:00+00:00', '2025-02-11 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=2609, freq='B')


In [50]:
# Find the missing dates by comparing the set of all workdays with the available dates in the 'Shanghai_Composite_data' index
Shanghai_Composite_missing_dates = Shanghai_Composite_all_workdays.difference(Shanghai_Composite_data.index)
print("Missing workdays:\n", Shanghai_Composite_missing_dates)

Missing workdays:
 DatetimeIndex(['2015-02-18 00:00:00+00:00', '2015-02-19 00:00:00+00:00',
               '2015-02-20 00:00:00+00:00', '2015-02-23 00:00:00+00:00',
               '2015-02-24 00:00:00+00:00', '2015-04-06 00:00:00+00:00',
               '2015-05-01 00:00:00+00:00', '2015-06-22 00:00:00+00:00',
               '2015-09-03 00:00:00+00:00', '2015-09-04 00:00:00+00:00',
               ...
               '2024-10-03 00:00:00+00:00', '2024-10-04 00:00:00+00:00',
               '2024-10-07 00:00:00+00:00', '2025-01-01 00:00:00+00:00',
               '2025-01-28 00:00:00+00:00', '2025-01-29 00:00:00+00:00',
               '2025-01-30 00:00:00+00:00', '2025-01-31 00:00:00+00:00',
               '2025-02-03 00:00:00+00:00', '2025-02-04 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=185, freq=None)


In [51]:
# Print the number of missing workdays by getting the length of the 'missing_dates' array
print("Number of missing workdays over the entire time range:", Shanghai_Composite_missing_dates.shape[0])

Number of missing workdays over the entire time range: 185


## Forward Filling the Missing Data

In [52]:
# Forward fill missing dates for all 7 foreign stock indices using the existing missing_dates variables

# List of tuples: (DataFrame variable, missing_dates variable)
index_info = [
    (CAC_data, CAC_missing_dates),
    (DAX_data, DAX_missing_dates),
    (FTSE_data, FTSE_missing_dates),
    (Hang_Seng_data, Hang_Seng_missing_dates),
    (NIFTY_data, NIFTY_missing_dates),
    (Nikkei_data, Nikkei_missing_dates),
    (Shanghai_Composite_data, Shanghai_Composite_missing_dates)
]

for i, (df, missing_dates) in enumerate(index_info):
    # Combine current index with missing dates, sort, and reindex
    all_dates = df.index.union(missing_dates).sort_values()
    df_reindexed = df.reindex(all_dates).ffill()
    # Assign back to the original variable
    if i == 0:
        CAC_data = df_reindexed
    elif i == 1:
        DAX_data = df_reindexed
    elif i == 2:
        FTSE_data = df_reindexed
    elif i == 3:
        Hang_Seng_data = df_reindexed
    elif i == 4:
        NIFTY_data = df_reindexed
    elif i == 5:
        Nikkei_data = df_reindexed
    elif i == 6:
        Shanghai_Composite_data = df_reindexed

CAC_data.head(10)

,Open,Close,Adj Close
2015-02-12 00:00:00+00:00,4670.080078,4726.200195,4726.200195
2015-02-13 00:00:00+00:00,4748.040039,4759.359863,4759.359863
2015-02-16 00:00:00+00:00,4757.520020,4751.950195,4751.950195
2015-02-17 00:00:00+00:00,4720.930176,4753.990234,4753.990234
2015-02-18 00:00:00+00:00,4782.589844,4799.029785,4799.029785
2015-02-19 00:00:00+00:00,4788.399902,4833.279785,4833.279785
2015-02-20 00:00:00+00:00,4821.379883,4830.899902,4830.899902
2015-02-23 00:00:00+00:00,4872.129883,4862.299805,4862.299805
2015-02-24 00:00:00+00:00,4865.680176,4886.439941,4886.439941
2015-02-25 00:00:00+00:00,4885.000000,4882.220215,4882.220215


## Calculating Daily Percentage Change

In [53]:
# Calculate the daily percentage change for CAC 40
CAC_data["CAC_Pct_Change"] = (((CAC_data["Adj Close"] - CAC_data["Adj Close"].shift()) / CAC_data["Close"].shift()) * 100).round(6)
CAC_data.head()

,Open,Close,Adj Close,CAC_Pct_Change
2015-02-12 00:00:00+00:00,4670.080078,4726.200195,4726.200195,NaN
2015-02-13 00:00:00+00:00,4748.040039,4759.359863,4759.359863,0.701614
2015-02-16 00:00:00+00:00,4757.520020,4751.950195,4751.950195,-0.155686
2015-02-17 00:00:00+00:00,4720.930176,4753.990234,4753.990234,0.042931
2015-02-18 00:00:00+00:00,4782.589844,4799.029785,4799.029785,0.947405


In [54]:
# Calculate the daily percentage change for DAX 40
DAX_data["DAX_Pct_Change"] = (((DAX_data["Adj Close"] - DAX_data["Adj Close"].shift()) / DAX_data["Close"].shift()) * 100).round(6)
DAX_data.head()

,Open,Close,Adj Close,DAX_Pct_Change
2015-02-12 00:00:00+00:00,10746.910156,10919.650391,10919.650391,NaN
2015-02-13 00:00:00+00:00,10952.389648,10963.400391,10963.400391,0.400654
2015-02-16 00:00:00+00:00,10946.879883,10923.230469,10923.230469,-0.366400
2015-02-17 00:00:00+00:00,10847.730469,10895.620117,10895.620117,-0.252767
2015-02-18 00:00:00+00:00,10931.719727,10961.000000,10961.000000,0.600057


In [55]:
# Calculate the daily percentage change for FTSE 100
FTSE_data["FTSE_Pct_Change"] = (((FTSE_data["Adj Close"] - FTSE_data["Adj Close"].shift()) / FTSE_data["Close"].shift()) * 100).round(6)
FTSE_data.head()

,Open,Close,Adj Close,FTSE_Pct_Change
2015-02-12 00:00:00+00:00,6818.200195,6828.100098,6828.100098,NaN
2015-02-13 00:00:00+00:00,6828.100098,6873.500000,6873.500000,0.664898
2015-02-16 00:00:00+00:00,6873.500000,6857.100098,6857.100098,-0.238596
2015-02-17 00:00:00+00:00,6857.100098,6898.100098,6898.100098,0.597920
2015-02-18 00:00:00+00:00,6898.100098,6898.100098,6898.100098,0.000000


In [56]:
# Calculate the daily percentage change for Hang Seng
Hang_Seng_data["HS_Pct_Change"] = (((Hang_Seng_data["Adj Close"] - Hang_Seng_data["Adj Close"].shift()) / Hang_Seng_data["Close"].shift()) * 100).round(6)
Hang_Seng_data.head()

,Open,Close,Adj Close,HS_Pct_Change
2015-02-12 00:00:00+00:00,24377.589844,24422.150391,24422.150391,NaN
2015-02-13 00:00:00+00:00,24535.660156,24682.539062,24682.539062,1.066199
2015-02-16 00:00:00+00:00,24717.490234,24726.529297,24726.529297,0.178224
2015-02-17 00:00:00+00:00,24653.880859,24784.880859,24784.880859,0.235988
2015-02-18 00:00:00+00:00,24836.050781,24832.080078,24832.080078,0.190436


In [57]:
# Calculate the daily percentage change for NIFTY 50
NIFTY_data["NIFTY_Pct_Change"] = (((NIFTY_data["Adj Close"] - NIFTY_data["Adj Close"].shift()) / NIFTY_data["Close"].shift()) * 100).round(6)
NIFTY_data.head()

,Open,Close,Adj Close,NIFTY_Pct_Change
2015-02-12 00:00:00+00:00,8676.950195,8711.549805,8711.549805,NaN
2015-02-13 00:00:00+00:00,8741.500000,8805.500000,8805.500000,1.078456
2015-02-16 00:00:00+00:00,8831.400391,8809.349609,8809.349609,0.043718
2015-02-17 00:00:00+00:00,8831.400391,8809.349609,8809.349609,0.000000
2015-02-18 00:00:00+00:00,8811.549805,8869.099609,8869.099609,0.678257


In [58]:
# Calculate the daily percentage change for Nikkei 225
Nikkei_data["Nikkei_Pct_Change"] = (((Nikkei_data["Adj Close"] - Nikkei_data["Adj Close"].shift()) / Nikkei_data["Close"].shift()) * 100).round(6)
Nikkei_data.head()

,Open,Close,Adj Close,Nikkei_Pct_Change
2015-02-12 00:00:00+00:00,17899.400391,17979.720703,17979.720703,NaN
2015-02-13 00:00:00+00:00,17892.099609,17913.359375,17913.359375,-0.369090
2015-02-16 00:00:00+00:00,18024.009766,18004.769531,18004.769531,0.510290
2015-02-17 00:00:00+00:00,17949.699219,17987.089844,17987.089844,-0.098194
2015-02-18 00:00:00+00:00,18103.980469,18199.169922,18199.169922,1.179068


In [59]:
# Calculate the daily percentage change for Shanghai Composite
Shanghai_Composite_data["Shanghai_Pct_Change"] = (((Shanghai_Composite_data["Adj Close"] - Shanghai_Composite_data["Adj Close"].shift()) / Shanghai_Composite_data["Close"].shift()) * 100).round(6)
Shanghai_Composite_data.head()

,Open,Close,Adj Close,Shanghai_Pct_Change
2015-02-12 00:00:00+00:00,3157.958984,3173.416016,3173.416016,NaN
2015-02-13 00:00:00+00:00,3186.808105,3203.826904,3203.826904,0.958301
2015-02-16 00:00:00+00:00,3206.136963,3222.363037,3222.363037,0.578562
2015-02-17 00:00:00+00:00,3230.884033,3246.906006,3246.906006,0.761645
2015-02-18 00:00:00+00:00,3230.884033,3246.906006,3246.906006,0.000000


## Computing 30-Day Rolling Standard Deviation


In [60]:
# Calculate the 30-day rolling standard deviation for CAC 40
CAC_data["CAC_30D_Rolling_Std"] = CAC_data["CAC_Pct_Change"].rolling(window=30).std().round(6)
CAC_data.head()

,Open,Close,Adj Close,CAC_Pct_Change,CAC_30D_Rolling_Std
2015-02-12 00:00:00+00:00,4670.080078,4726.200195,4726.200195,NaN,NaN
2015-02-13 00:00:00+00:00,4748.040039,4759.359863,4759.359863,0.701614,NaN
2015-02-16 00:00:00+00:00,4757.520020,4751.950195,4751.950195,-0.155686,NaN
2015-02-17 00:00:00+00:00,4720.930176,4753.990234,4753.990234,0.042931,NaN
2015-02-18 00:00:00+00:00,4782.589844,4799.029785,4799.029785,0.947405,NaN


In [61]:

# Calculate the 30-day rolling standard deviation for DAX 40
DAX_data["DAX_30D_Rolling_Std"] = DAX_data["DAX_Pct_Change"].rolling(window=30).std().round(6)
DAX_data.head()

,Open,Close,Adj Close,DAX_Pct_Change,DAX_30D_Rolling_Std
2015-02-12 00:00:00+00:00,10746.910156,10919.650391,10919.650391,NaN,NaN
2015-02-13 00:00:00+00:00,10952.389648,10963.400391,10963.400391,0.400654,NaN
2015-02-16 00:00:00+00:00,10946.879883,10923.230469,10923.230469,-0.366400,NaN
2015-02-17 00:00:00+00:00,10847.730469,10895.620117,10895.620117,-0.252767,NaN
2015-02-18 00:00:00+00:00,10931.719727,10961.000000,10961.000000,0.600057,NaN


In [62]:
# Calculate the 30-day rolling standard deviation for FTSE 100
FTSE_data["FTSE_30D_Rolling_Std"] = FTSE_data["FTSE_Pct_Change"].rolling(window=30).std().round(6)
FTSE_data.head()

,Open,Close,Adj Close,FTSE_Pct_Change,FTSE_30D_Rolling_Std
2015-02-12 00:00:00+00:00,6818.200195,6828.100098,6828.100098,NaN,NaN
2015-02-13 00:00:00+00:00,6828.100098,6873.500000,6873.500000,0.664898,NaN
2015-02-16 00:00:00+00:00,6873.500000,6857.100098,6857.100098,-0.238596,NaN
2015-02-17 00:00:00+00:00,6857.100098,6898.100098,6898.100098,0.597920,NaN
2015-02-18 00:00:00+00:00,6898.100098,6898.100098,6898.100098,0.000000,NaN


In [63]:
# Calculate the 30-day rolling standard deviation for Hang Seng
Hang_Seng_data["HS_30D_Rolling_Std"] = Hang_Seng_data["HS_Pct_Change"].rolling(window=30).std().round(6)
Hang_Seng_data.head()

,Open,Close,Adj Close,HS_Pct_Change,HS_30D_Rolling_Std
2015-02-12 00:00:00+00:00,24377.589844,24422.150391,24422.150391,NaN,NaN
2015-02-13 00:00:00+00:00,24535.660156,24682.539062,24682.539062,1.066199,NaN
2015-02-16 00:00:00+00:00,24717.490234,24726.529297,24726.529297,0.178224,NaN
2015-02-17 00:00:00+00:00,24653.880859,24784.880859,24784.880859,0.235988,NaN
2015-02-18 00:00:00+00:00,24836.050781,24832.080078,24832.080078,0.190436,NaN


In [64]:
# Calculate the 30-day rolling standard deviation for NIFTY 50
NIFTY_data["NIFTY_30D_Rolling_Std"] = NIFTY_data["NIFTY_Pct_Change"].rolling(window=30).std().round(6)
NIFTY_data.head()

,Open,Close,Adj Close,NIFTY_Pct_Change,NIFTY_30D_Rolling_Std
2015-02-12 00:00:00+00:00,8676.950195,8711.549805,8711.549805,NaN,NaN
2015-02-13 00:00:00+00:00,8741.500000,8805.500000,8805.500000,1.078456,NaN
2015-02-16 00:00:00+00:00,8831.400391,8809.349609,8809.349609,0.043718,NaN
2015-02-17 00:00:00+00:00,8831.400391,8809.349609,8809.349609,0.000000,NaN
2015-02-18 00:00:00+00:00,8811.549805,8869.099609,8869.099609,0.678257,NaN


In [65]:
# Calculate the 30-day rolling standard deviation for Nikkei 225
Nikkei_data["Nikkei_30D_Rolling_Std"] = Nikkei_data["Nikkei_Pct_Change"].rolling(window=30).std().round(6)
Nikkei_data.head()

,Open,Close,Adj Close,Nikkei_Pct_Change,Nikkei_30D_Rolling_Std
2015-02-12 00:00:00+00:00,17899.400391,17979.720703,17979.720703,NaN,NaN
2015-02-13 00:00:00+00:00,17892.099609,17913.359375,17913.359375,-0.369090,NaN
2015-02-16 00:00:00+00:00,18024.009766,18004.769531,18004.769531,0.510290,NaN
2015-02-17 00:00:00+00:00,17949.699219,17987.089844,17987.089844,-0.098194,NaN
2015-02-18 00:00:00+00:00,18103.980469,18199.169922,18199.169922,1.179068,NaN


In [66]:
# Calculate the 30-day rolling standard deviation for Shanghai Composite
Shanghai_Composite_data["Shanghai_30D_Rolling_Std"] = Shanghai_Composite_data["Shanghai_Pct_Change"].rolling(window=30).std().round(6)
Shanghai_Composite_data.head()

,Open,Close,Adj Close,Shanghai_Pct_Change,Shanghai_30D_Rolling_Std
2015-02-12 00:00:00+00:00,3157.958984,3173.416016,3173.416016,NaN,NaN
2015-02-13 00:00:00+00:00,3186.808105,3203.826904,3203.826904,0.958301,NaN
2015-02-16 00:00:00+00:00,3206.136963,3222.363037,3222.363037,0.578562,NaN
2015-02-17 00:00:00+00:00,3230.884033,3246.906006,3246.906006,0.761645,NaN
2015-02-18 00:00:00+00:00,3230.884033,3246.906006,3246.906006,0.000000,NaN


## Creating a Shock Column


In [67]:
# Create a shock column for CAC 40
CAC_data['Shock'] = (CAC_data['CAC_Pct_Change'].abs() > 1.5 * CAC_data['CAC_30D_Rolling_Std']).astype(int)
CAC_data.tail()

,Open,Close,Adj Close,CAC_Pct_Change,CAC_30D_Rolling_Std,Shock
2025-02-05 00:00:00+00:00,7888.600098,7891.680176,7891.680176,-0.186175,0.821051,0
2025-02-06 00:00:00+00:00,7918.359863,8007.620117,8007.620117,1.469141,0.847692,1
2025-02-07 00:00:00+00:00,7998.379883,7973.029785,7973.029785,-0.431968,0.848473,0
2025-02-10 00:00:00+00:00,7977.759766,8006.220215,8006.220215,0.416284,0.833734,0
2025-02-11 00:00:00+00:00,8013.009766,8028.899902,8028.899902,0.283276,0.825652,0


In [68]:
# Create a shock column for DAX 40
DAX_data['Shock'] = (DAX_data['DAX_Pct_Change'].abs() > 1.5 * DAX_data['DAX_30D_Rolling_Std']).astype(int)
DAX_data.tail()

,Open,Close,Adj Close,DAX_Pct_Change,DAX_30D_Rolling_Std,Shock
2025-02-05 00:00:00+00:00,21429.830078,21585.929688,21585.929688,0.373066,0.652402,0
2025-02-06 00:00:00+00:00,21718.099609,21902.419922,21902.419922,1.466188,0.684660,1
2025-02-07 00:00:00+00:00,21908.480469,21787.000000,21787.000000,-0.526973,0.698705,0
2025-02-10 00:00:00+00:00,21830.929688,21911.740234,21911.740234,0.572544,0.688883,0
2025-02-11 00:00:00+00:00,21909.960938,22037.830078,22037.830078,0.575444,0.687610,0


In [69]:
# Create a shock column for FTSE 100
FTSE_data['Shock'] = (FTSE_data['FTSE_Pct_Change'].abs() > 1.5 * FTSE_data['FTSE_30D_Rolling_Std']).astype(int)
FTSE_data.tail()

,Open,Close,Adj Close,FTSE_Pct_Change,FTSE_30D_Rolling_Std,Shock
2025-02-05 00:00:00+00:00,8570.799805,8623.299805,8623.299805,0.612545,0.602648,0
2025-02-06 00:00:00+00:00,8623.299805,8727.299805,8727.299805,1.206035,0.628820,1
2025-02-07 00:00:00+00:00,8727.299805,8700.500000,8700.500000,-0.307080,0.636484,0
2025-02-10 00:00:00+00:00,8700.500000,8767.799805,8767.799805,0.773517,0.634719,0
2025-02-11 00:00:00+00:00,8767.799805,8777.400391,8777.400391,0.109498,0.631074,0


In [70]:
# Create a shock column for Hang Seng
Hang_Seng_data['Shock'] = (Hang_Seng_data['HS_Pct_Change'].abs() > 1.5 * Hang_Seng_data['HS_30D_Rolling_Std']).astype(int)
Hang_Seng_data.tail()

,Open,Close,Adj Close,HS_Pct_Change,HS_30D_Rolling_Std,Shock
2025-02-05 00:00:00+00:00,20827.769531,20597.089844,20597.089844,-0.927713,1.080463,0
2025-02-06 00:00:00+00:00,20598.980469,20891.619141,20891.619141,1.429956,1.107675,0
2025-02-07 00:00:00+00:00,20858.519531,21133.539062,21133.539062,1.157976,1.122647,0
2025-02-10 00:00:00+00:00,21223.609375,21521.980469,21521.980469,1.838033,1.159601,1
2025-02-11 00:00:00+00:00,21656.820312,21294.859375,21294.859375,-1.055298,1.183472,0


In [71]:
# Create a shock column for NIFTY 50
NIFTY_data['Shock'] = (NIFTY_data['NIFTY_Pct_Change'].abs() > 1.5 * NIFTY_data['NIFTY_30D_Rolling_Std']).astype(int)
NIFTY_data.tail()

,Open,Close,Adj Close,NIFTY_Pct_Change,NIFTY_30D_Rolling_Std,Shock
2025-02-05 00:00:00+00:00,23801.750000,23696.300781,23696.300781,-0.180921,0.845264,0
2025-02-06 00:00:00+00:00,23761.949219,23603.349609,23603.349609,-0.392260,0.848034,0
2025-02-07 00:00:00+00:00,23649.500000,23559.949219,23559.949219,-0.183874,0.846829,0
2025-02-10 00:00:00+00:00,23543.800781,23381.599609,23381.599609,-0.757003,0.848235,0
2025-02-11 00:00:00+00:00,23383.550781,23071.800781,23071.800781,-1.324968,0.880301,1


In [72]:
# Create a shock column for Nikkei 225
Nikkei_data['Shock'] = (Nikkei_data['Nikkei_Pct_Change'].abs() > 1.5 * Nikkei_data['Nikkei_30D_Rolling_Std']).astype(int)
Nikkei_data.tail()

,Open,Close,Adj Close,Nikkei_Pct_Change,Nikkei_30D_Rolling_Std,Shock
2025-02-04 00:00:00+00:00,39078.660156,38798.371094,38798.371094,0.722431,1.070622,0
2025-02-05 00:00:00+00:00,38979.210938,38831.480469,38831.480469,0.085337,1.069726,0
2025-02-06 00:00:00+00:00,38977.578125,39066.531250,39066.531250,0.605310,1.054897,0
2025-02-07 00:00:00+00:00,38907.761719,38787.019531,38787.019531,-0.715476,1.002362,0
2025-02-10 00:00:00+00:00,38736.371094,38801.171875,38801.171875,0.036487,0.990043,0


In [73]:
# Create a shock column for Shanghai Composite
Shanghai_Composite_data['Shock'] = (Shanghai_Composite_data['Shanghai_Pct_Change'].abs() > 1.5 * Shanghai_Composite_data['Shanghai_30D_Rolling_Std']).astype(int)
Shanghai_Composite_data.tail()

,Open,Close,Adj Close,Shanghai_Pct_Change,Shanghai_30D_Rolling_Std,Shock
2025-02-05 00:00:00+00:00,3270.159912,3229.488037,3229.488037,-0.649512,0.888607,0
2025-02-06 00:00:00+00:00,3226.574951,3270.658936,3270.658936,1.274843,0.925269,0
2025-02-07 00:00:00+00:00,3269.227051,3303.666992,3303.666992,1.009217,0.947702,0
2025-02-10 00:00:00+00:00,3308.040039,3322.169922,3322.169922,0.560072,0.953666,0
2025-02-11 00:00:00+00:00,3323.668945,3318.056885,3318.056885,-0.123806,0.907749,0


In [74]:
# Drop the first 30 rows from each DataFrame to remove NaN values from the rolling calculations
CAC_data = CAC_data.iloc[30:]
DAX_data = DAX_data.iloc[30:]
FTSE_data = FTSE_data.iloc[30:]
Hang_Seng_data = Hang_Seng_data.iloc[30:]
NIFTY_data = NIFTY_data.iloc[30:]
Nikkei_data = Nikkei_data.iloc[30:]
Shanghai_Composite_data = Shanghai_Composite_data.iloc[30:]

## Merging the DataFrames


In [75]:
# CAC
CAC_tidy = CAC_data[['Open', 'Close', 'Adj Close', 'CAC_30D_Rolling_Std', 'CAC_Pct_Change', 'Shock']]
CAC_tidy = CAC_tidy.rename(columns={
    'CAC_30D_Rolling_Std': '30D_Rolling_Std',
    'CAC_Pct_Change': 'Pct_Change'
})
CAC_tidy['Index'] = 'CAC'

In [76]:
# DAX
DAX_tidy = DAX_data[['Open', 'Close', 'Adj Close', 'DAX_30D_Rolling_Std', 'DAX_Pct_Change', 'Shock']]
DAX_tidy = DAX_tidy.rename(columns={
    'DAX_30D_Rolling_Std': '30D_Rolling_Std',
    'DAX_Pct_Change': 'Pct_Change'
})
DAX_tidy['Index'] = 'DAX'

In [77]:
# FTSE
FTSE_tidy = FTSE_data[['Open', 'Close', 'Adj Close', 'FTSE_30D_Rolling_Std', 'FTSE_Pct_Change', 'Shock']]
FTSE_tidy = FTSE_tidy.rename(columns={
    'FTSE_30D_Rolling_Std': '30D_Rolling_Std',
    'FTSE_Pct_Change': 'Pct_Change'
})
FTSE_tidy['Index'] = 'FTSE'

In [78]:
# Hang Seng
Hang_Seng_tidy = Hang_Seng_data[['Open', 'Close', 'Adj Close', 'HS_30D_Rolling_Std', 'HS_Pct_Change', 'Shock']]
Hang_Seng_tidy = Hang_Seng_tidy.rename(columns={
    'HS_30D_Rolling_Std': '30D_Rolling_Std',
    'HS_Pct_Change': 'Pct_Change'
})
Hang_Seng_tidy['Index'] = 'Hang Seng'

In [79]:
# NIFTY
NIFTY_tidy = NIFTY_data[['Open', 'Close', 'Adj Close', 'NIFTY_30D_Rolling_Std', 'NIFTY_Pct_Change', 'Shock']]
NIFTY_tidy = NIFTY_tidy.rename(columns={
    'NIFTY_30D_Rolling_Std': '30D_Rolling_Std',
    'NIFTY_Pct_Change': 'Pct_Change'
})
NIFTY_tidy['Index'] = 'NIFTY'

In [80]:
# Nikkei
Nikkei_tidy = Nikkei_data[['Open', 'Close', 'Adj Close', 'Nikkei_30D_Rolling_Std', 'Nikkei_Pct_Change', 'Shock']]
Nikkei_tidy = Nikkei_tidy.rename(columns={
    'Nikkei_30D_Rolling_Std': '30D_Rolling_Std',
    'Nikkei_Pct_Change': 'Pct_Change'
})
Nikkei_tidy['Index'] = 'Nikkei'

In [81]:
# Shanghai Composite
Shanghai_Composite_tidy = Shanghai_Composite_data[['Open', 'Close', 'Adj Close', 'Shanghai_30D_Rolling_Std', 'Shanghai_Pct_Change', 'Shock']]
Shanghai_Composite_tidy = Shanghai_Composite_tidy.rename(columns={
    'Shanghai_30D_Rolling_Std': '30D_Rolling_Std',
    'Shanghai_Pct_Change': 'Pct_Change'
})
Shanghai_Composite_tidy['Index'] = 'Shanghai Composite'


In [82]:
# Concatenate all tidy DataFrames into one
preprocessed_data = pd.concat([CAC_tidy, DAX_tidy, FTSE_tidy, Hang_Seng_tidy, NIFTY_tidy, Nikkei_tidy, Shanghai_Composite_tidy])
preprocessed_data.head()

,Open,Close,Adj Close,30D_Rolling_Std,Pct_Change,Shock,Index
2015-03-26 00:00:00+00:00,4992.549805,5006.350098,5006.350098,0.795626,-0.291579,0,CAC
2015-03-27 00:00:00+00:00,5020.089844,5034.060059,5034.060059,0.792829,0.553496,0,CAC
2015-03-30 00:00:00+00:00,5052.250000,5083.520020,5083.520020,0.802874,0.982506,0,CAC
2015-03-31 00:00:00+00:00,5073.990234,5033.640137,5033.640137,0.832258,-0.981208,0,CAC
2015-04-01 00:00:00+00:00,5009.629883,5062.220215,5062.220215,0.823246,0.567782,0,CAC


In [83]:
# Format the 'Date' index
preprocessed_data.index = pd.to_datetime(preprocessed_data.index).normalize()

# Reorder the columns to have 'Index' as the second column
preprocessed_data = preprocessed_data[['Index', 'Open', 'Close', 'Adj Close', '30D_Rolling_Std', 'Pct_Change', 'Shock']]

## Cleaning and Preprocessing the NASDAQ 100 Index Data

In [84]:
# Load NASDAQ-100 data
NASDAQ_data = pd.read_csv('Data/NASDAQ100.csv', index_col="Date")

# Convert index to datetime
NASDAQ_data.index = pd.to_datetime(NASDAQ_data.index, utc=True)

NASDAQ_data.head()


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2015-02-17 05:00:00+00:00,4379.319824,4387.319824,4370.629883,4385.339844,1751490000,0.0,0.0
2015-02-18 05:00:00+00:00,4378.729980,4393.189941,4374.229980,4390.910156,1684800000,0.0,0.0
2015-02-19 05:00:00+00:00,4388.919922,4415.790039,4388.439941,4411.859863,1592890000,0.0,0.0
2015-02-20 05:00:00+00:00,4406.750000,4444.620117,4397.620117,4443.049805,1761140000,0.0,0.0
2015-02-23 05:00:00+00:00,4441.259766,4449.490234,4430.870117,4449.490234,1754850000,0.0,0.0


In [85]:
# Drop unnecessary columns
NASDAQ_data = NASDAQ_data.drop(columns=['High', 'Low', 'Volume', 'Dividends', 'Stock Splits'])
NASDAQ_data.head()

,Open,Close
Date,,
2015-02-17 05:00:00+00:00,4379.319824,4385.339844
2015-02-18 05:00:00+00:00,4378.729980,4390.910156
2015-02-19 05:00:00+00:00,4388.919922,4411.859863
2015-02-20 05:00:00+00:00,4406.750000,4443.049805
2015-02-23 05:00:00+00:00,4441.259766,4449.490234


### Checking for Missing Values in NASDAQ 100

In [86]:
# Check for NaN values in the dataset
missing_values = NASDAQ_data.isna().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Open     0
Close    0
dtype: int64


In [87]:
# Find the initial date of the dataset
NASDAQ_start = NASDAQ_data.index.min()
print(NASDAQ_start)

2015-02-17 05:00:00+00:00


In [88]:
# Find the end date of the dataset
NASDAQ_end = NASDAQ_data.index.max()
print(NASDAQ_end)

2025-02-14 05:00:00+00:00


In [89]:
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

# Normalize both to remove time-of-day component
NASDAQ_data.index = NASDAQ_data.index.normalize()

# Define US market trading days
us_bd = CustomBusinessDay(calendar=USFederalHolidayCalendar())
expected_days = pd.date_range(start=NASDAQ_start, end=NASDAQ_end, freq=us_bd)
expected_days = expected_days.normalize()

# Compare against actual data
missing_days = expected_days.difference(NASDAQ_data.index)

print(f"True data gaps (after excluding holidays): {len(missing_days)}")
missing_days[:10]


True data gaps (after excluding holidays): 12


DatetimeIndex(['2015-04-03 00:00:00+00:00', '2016-03-25 00:00:00+00:00',
               '2017-04-14 00:00:00+00:00', '2018-03-30 00:00:00+00:00',
               '2018-12-05 00:00:00+00:00', '2019-04-19 00:00:00+00:00',
               '2020-04-10 00:00:00+00:00', '2021-04-02 00:00:00+00:00',
               '2022-04-15 00:00:00+00:00', '2023-04-07 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', freq=None)

The identified missing values are all 'Good Friday' holidays, which are not trading days for the NASDAQ 100 index but also not in the official US Federal Holidays calendar. The date '2018-12-05' was identified as a special closure day since the market was closed due to a national day of mourning for President George H.W. Bush.


In [90]:
# Calculate previous day's Close
NASDAQ_data['Prev_Close'] = (NASDAQ_data['Close'].shift(1)).round(7)

# Calculate the opening gap
NASDAQ_data['Opening_Gap'] = ((NASDAQ_data['Open'] - NASDAQ_data['Prev_Close']) / NASDAQ_data['Prev_Close'] * 100).round(7)

# Define classification labels based on gap thresholds
GAP_THRESHOLD = 0.4

def classify_gap(gap):
    if gap >= GAP_THRESHOLD:
        return "Up"
    elif gap <= -GAP_THRESHOLD:
        return "Down"
    else:
        return "Flat"


NASDAQ_data['Gap_Direction'] = NASDAQ_data['Opening_Gap'].apply(classify_gap)

# Daily return
NASDAQ_data['Daily_Return'] = ((NASDAQ_data['Close'] - NASDAQ_data['Open']) / NASDAQ_data['Open'] * 100).round(7)

# Preview
NASDAQ_data.head(10)


,Open,Close,Prev_Close,Opening_Gap,Gap_Direction,Daily_Return
Date,,,,,,
2015-02-17 00:00:00+00:00,4379.319824,4385.339844,NaN,NaN,Flat,0.137465
2015-02-18 00:00:00+00:00,4378.729980,4390.910156,4385.339844,-0.150726,Flat,0.278167
2015-02-19 00:00:00+00:00,4388.919922,4411.859863,4390.910156,-0.045326,Flat,0.522679
2015-02-20 00:00:00+00:00,4406.750000,4443.049805,4411.859863,-0.115821,Flat,0.823732
2015-02-23 00:00:00+00:00,4441.259766,4449.490234,4443.049805,-0.040288,Flat,0.185318
2015-02-24 00:00:00+00:00,4442.129883,4451.029785,4449.490234,-0.165420,Flat,0.200352
2015-02-25 00:00:00+00:00,4441.080078,4440.589844,4451.029785,-0.223537,Flat,-0.011039
2015-02-26 00:00:00+00:00,4443.549805,4462.270020,4440.589844,0.066657,Flat,0.421290
2015-02-27 00:00:00+00:00,4459.859863,4440.669922,4462.270019,-0.054012,Flat,-0.430281


### Aligning NASDAQ 100 Data with Other Indices


In [91]:
# # Load the preprocessed data
# preprocessed_data = pd.read_csv('Preprocessed_data.csv', index_col="Date", parse_dates=True)

# Both DataFrames have datetime indexes with normalized dates
NASDAQ_data.index = pd.to_datetime(NASDAQ_data.index, utc=True).normalize()
preprocessed_data.index = pd.to_datetime(preprocessed_data.index, utc=True).normalize()

# Find common dates
common_dates = NASDAQ_data.index.intersection(preprocessed_data.index)

# Keep only rows with common dates in both DataFrames
NASDAQ_data = NASDAQ_data.loc[common_dates]
preprocessed_data = preprocessed_data.loc[common_dates]

In [92]:
NASDAQ_data.head()

,Open,Close,Prev_Close,Opening_Gap,Gap_Direction,Daily_Return
2015-03-26 00:00:00+00:00,4288.160156,4315.259766,4329.290039,-0.950038,Down,0.631964
2015-03-27 00:00:00+00:00,4314.600098,4332.859863,4315.259766,-0.015287,Flat,0.423209
2015-03-30 00:00:00+00:00,4359.779785,4382.819824,4332.859863,0.621297,Up,0.528468
2015-03-31 00:00:00+00:00,4364.140137,4333.689941,4382.819824,-0.426202,Down,-0.697736
2015-04-01 00:00:00+00:00,4328.169922,4311.259766,4333.689941,-0.127375,Flat,-0.390700


In [93]:
preprocessed_data.head()

,Index,Open,Close,Adj Close,30D_Rolling_Std,Pct_Change,Shock
2015-03-26 00:00:00+00:00,CAC,4992.549805,5006.350098,5006.350098,0.795626,-0.291579,0
2015-03-26 00:00:00+00:00,DAX,11736.339844,11843.679688,11843.679688,0.942706,-0.182386,0
2015-03-26 00:00:00+00:00,FTSE,6991.000000,6895.299805,6895.299805,0.759011,-1.368906,1
2015-03-26 00:00:00+00:00,Hang Seng,24442.509766,24497.080078,24497.080078,0.593293,-0.126998,0
2015-03-26 00:00:00+00:00,NIFTY,8474.950195,8342.150391,8342.150391,0.903772,-2.211392,1


In [94]:
# Save the NASDAQ data to a CSV file
NASDAQ_data.to_csv('NASDAQ_Index_Preprocessed.csv')

# Save the preprocessed data to a CSV file
preprocessed_data.to_csv('Preprocessed_Indices_Data.csv')

In [95]:
# Pivot percent changes and shocks
pct_wide = preprocessed_data.pivot_table(values="Pct_Change", index=preprocessed_data.index, columns="Index")
pct_wide.columns = [f"{c}_pct" for c in pct_wide.columns]

shock_wide = preprocessed_data.pivot_table(values="Shock", index=preprocessed_data.index, columns="Index")
shock_wide.columns = [f"{c}_shock" for c in shock_wide.columns]

# Combine
features = pct_wide.join(shock_wide)

# Shift European markets by 1 day to reflect time difference
euro_cols = [c for c in features.columns if c.startswith(("DAX_", "CAC_", "FTSE_"))]
features[euro_cols] = features[euro_cols].shift(1)

# Weighted global movement
features['Weighted_Global_Move'] = (
    0.4 * features['NIFTY_pct'] +
    0.3 * features['Hang Seng_pct'] +
    0.15 * features['Nikkei_pct'] +
    0.05 * features['Shanghai Composite_pct'] +
    0.05 * features['FTSE_pct'] +
    0.025 * features['DAX_pct'] +
    0.025 * features['CAC_pct']
)


# Weighted shock score
features['Weighted_Shock_Score'] = (
    0.4 * features['NIFTY_shock'] +
    0.3 * features['Hang Seng_shock'] +
    0.15 * features['Nikkei_shock'] +
    0.05 * features['Shanghai Composite_shock'] +
    0.05 * features['FTSE_shock'] +
    0.025 * features['DAX_shock'] +
    0.025 * features['CAC_shock']
)


# Keep these after your initial pivot + weighting steps

# Rolling volatility features (short term only)
features['Global_Volatility_5d'] = features['Weighted_Global_Move'].rolling(window=5).std()

# Regional divergence
features['Europe_vs_Asia'] = (
    (features['DAX_pct'] + features['FTSE_pct'] + features['CAC_pct']) / 3 -
    (features['Nikkei_pct'] + features['NIFTY_pct'] + features['Hang Seng_pct'] + features['Shanghai Composite_pct']) / 4
)

# Drop NaNs
features = features.dropna()
X = features

y = NASDAQ_data["Gap_Direction"].loc[features.index]


print(X.shape, y.shape)
print(y.value_counts())


(2480, 18) (2480,)
Gap_Direction
Flat    1299
Up       669
Down     512
Name: count, dtype: int64


In [96]:
X.head(10)

,CAC_pct,DAX_pct,FTSE_pct,Hang Seng_pct,NIFTY_pct,Nikkei_pct,Shanghai Composite_pct,CAC_shock,DAX_shock,FTSE_shock,Hang Seng_shock,NIFTY_shock,Nikkei_shock,Shanghai Composite_shock,Weighted_Global_Move,Weighted_Shock_Score,Global_Volatility_5d,Europe_vs_Asia
2015-04-02 00:00:00+00:00,0.567782,0.294246,0.538904,0.769017,0.000000,1.460213,0.406530,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.518559,0.150,0.634895,-0.191963
2015-04-06 00:00:00+00:00,0.235468,-0.283219,0.352449,0.000000,0.857771,-0.190890,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.330904,0.000,0.533318,-0.065154
2015-04-07 00:00:00+00:00,0.000000,0.000000,0.000000,0.000000,0.004612,1.250432,2.522018,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.315511,0.200,0.230320,-0.944265
2015-04-08 00:00:00+00:00,1.518480,1.304628,1.877512,3.802945,0.624696,0.760017,0.843977,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.711417,0.375,0.588153,0.058965
2015-04-09 00:00:00+00:00,-0.278190,-0.723051,-0.350483,2.696707,0.733262,0.747406,-0.933138,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.125216,0.300,0.606308,-1.261634
2015-04-10 00:00:00+00:00,1.403393,1.084925,1.124341,1.217322,0.023351,-0.150919,1.939999,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.567324,0.075,0.601011,0.446781
2015-04-13 00:00:00+00:00,0.604916,1.712005,1.060523,2.727847,0.611028,-0.010900,2.166536,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.280406,0.375,0.560392,-0.247813
2015-04-14 00:00:00+00:00,0.260667,-0.290915,-0.359672,-1.623515,0.000000,0.016170,0.336028,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.486567,0.000,0.846877,0.187856
2015-04-15 00:00:00+00:00,-0.686320,-0.900667,0.155713,0.208008,0.000000,-0.195492,-1.242922,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.060957,0.000,0.756858,-0.169490
2015-04-16 00:00:00+00:00,0.695470,0.030588,0.303874,0.437711,-1.441021,0.080574,2.709494,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.264189,0.450,0.717093,-0.103379


In [97]:
y.head(10)

2015-04-02 00:00:00+00:00    Flat
2015-04-06 00:00:00+00:00    Down
2015-04-07 00:00:00+00:00    Flat
2015-04-08 00:00:00+00:00    Flat
2015-04-09 00:00:00+00:00    Flat
2015-04-10 00:00:00+00:00    Flat
2015-04-13 00:00:00+00:00    Flat
2015-04-14 00:00:00+00:00    Flat
2015-04-15 00:00:00+00:00    Flat
2015-04-16 00:00:00+00:00    Flat
Name: Gap_Direction, dtype: object

This phase prepares the raw market data for modeling by cleaning, aligning, and transforming it into a usable structure. The main objective is to predict whether the NASDAQ-100 will open higher, lower, or flat based on movements in global stock markets that happen while the U.S. markets are closed.

The data from the NASDAQ-100 and several international indices (including DAX-40, CAC-40, FTSE-100, NIFTY-50, Nikkei-225, Hang Seng, and Shanghai Composite) was loaded with date information set as the index. All dates were converted to the same time zone (UTC) to ensure that movements in different markets could be compared on a shared timeline. This alignment is necessary because these markets operate in different time zones, and their movements can only affect the NASDAQ-100 on the following trading day.

Some days were missing from the data due to market holidays. To keep the data consistent, missing dates were forward-filled using the last available value. This prevents unnecessary gaps in the data without making unrealistic assumptions.

 For each index, the daily percentage change in adjusted closing price was calculated. This value reflects how much a market moved on a given day and is one of the main indicators used to understand overnight activity. A 30-day rolling standard deviation was then calculated to measure recent volatility. Based on this, a new feature called Shock was created. A day is labeled as a shock if the daily return was unusually large,specifically, more than 1.5 times the recent 30-day volatility. This helps identify days when something significant happened in a foreign market, which could influence the U.S. market.

The data was then reshaped so that each row represents one trading day, and each column contains data from one of the indices. This is necessary to train the model, which expects a fixed set of inputs for each prediction.

European markets are still open, or have only just closed, when the U.S. market opens, so their data was shifted forward by one day. This ensures that the model only uses the previous day's European close, which is what a trader would actually know at the opening bell, rather than data that would not be available yet.

Additional features were created to give a broader picture of market activity. For example, Weighted_Global_Move combines the daily changes from all indices, with more weight given to markets that are more influential. Another feature, Europe_vs_Asia, captures the difference between European and Asian market performance. These help the model understand larger patterns and imbalances between regions.

Lastly, the target variable (Gap_Direction) was defined based on how the NASDAQ-100 opened compared to the previous day’s close. If the opening was more than 0.25% higher, it was labeled "Up"; more than 0.25% lower, "Down"; otherwise, "Flat". This three-class label is what the model aims to predict.

These steps result in a clean and consistent dataset that reflects the project’s objective: to understand how overnight activity in global markets relates to the next-day opening of the NASDAQ-100.


## Step 4: Model development

In [98]:
# Prerequisites for machine learning
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

### Linear Regression

In [99]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import pandas as pd

X_lr = X.dropna()
y_lr = y.loc[X_lr.index]

le = LabelEncoder()
y_lr_enc = pd.Series(le.fit_transform(y_lr), index=y_lr.index)

split_point = int(len(X_lr) * 0.8)
X_train_lr, X_test_lr = X_lr.iloc[:split_point], X_lr.iloc[split_point:]
y_train_lr, y_test_lr = y_lr_enc.iloc[:split_point], y_lr_enc.iloc[split_point:]

scaler_lr = StandardScaler()
X_train_scaled_lr = scaler_lr.fit_transform(X_train_lr)
X_test_scaled_lr = scaler_lr.transform(X_test_lr)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled_lr, y_train_lr)

y_pred_lr = lr_model.predict(X_test_scaled_lr)
y_pred_class_lr = pd.Series(y_pred_lr.round().astype(int), index=y_test_lr.index)
y_pred_class_lr = y_pred_class_lr.clip(0, len(le.classes_) - 1)

print("Accuracy:", accuracy_score(y_test_lr, y_pred_class_lr))
print(classification_report(y_test_lr, y_pred_class_lr, target_names=le.classes_))

y_proba_lr = pd.get_dummies(y_pred_class_lr)
roc_auc_lr = roc_auc_score(y_test_lr, y_proba_lr, multi_class='ovr', average='macro')
print("ROC-AUC (macro, ovr):", roc_auc_lr)

Accuracy: 0.530241935483871
              precision    recall  f1-score   support

        Down       0.57      0.04      0.08        92
        Flat       0.54      0.95      0.68       265
          Up       0.39      0.05      0.09       139

    accuracy                           0.53       496
   macro avg       0.50      0.35      0.28       496
weighted avg       0.50      0.53      0.41       496

ROC-AUC (macro, ovr): 0.5097485589616698


### Random Forest

In [100]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import pandas as pd

X_rf = X.dropna()
y_rf = y.loc[X_rf.index]

le = LabelEncoder()
y_rf_enc = pd.Series(le.fit_transform(y_rf), index=y_rf.index)

split_point = int(len(X_rf) * 0.8)
X_train_rf, X_test_rf = X_rf.iloc[:split_point], X_rf.iloc[split_point:]
y_train_rf, y_test_rf = y_rf_enc.iloc[:split_point], y_rf_enc.iloc[split_point:]

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train_rf)

y_pred_rf = rf_model.predict(X_test_rf)
print("Accuracy:", accuracy_score(y_test_rf, y_pred_rf))
print(classification_report(y_test_rf, y_pred_rf, target_names=le.classes_))

y_proba_rf = rf_model.predict_proba(X_test_rf)
roc_auc_rf = roc_auc_score(y_test_rf, y_proba_rf, multi_class='ovr', average='macro')
print("ROC-AUC (macro, ovr):", roc_auc_rf)


Accuracy: 0.5141129032258065
              precision    recall  f1-score   support

        Down       0.43      0.30      0.36        92
        Flat       0.55      0.76      0.64       265
          Up       0.40      0.19      0.25       139

    accuracy                           0.51       496
   macro avg       0.46      0.42      0.42       496
weighted avg       0.49      0.51      0.48       496

ROC-AUC (macro, ovr): 0.6280039327400903


### XGBoost

In [101]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from collections import Counter
import pandas as pd

X_xgb = X.dropna()
y_xgb = y.loc[X_xgb.index]

le = LabelEncoder()
y_xgb_enc = pd.Series(le.fit_transform(y_xgb), index=y_xgb.index)

split_point = int(len(X_xgb) * 0.8)
X_train_xgb, X_test_xgb = X_xgb.iloc[:split_point], X_xgb.iloc[split_point:]
y_train_xgb, y_test_xgb = y_xgb_enc.iloc[:split_point], y_xgb_enc.iloc[split_point:]

counts = Counter(y_train_xgb)
weights = {cls: max(counts.values()) / cnt for cls, cnt in counts.items()}
sample_weights = y_train_xgb.map(weights)

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.075,
    min_child_weight=1,
    gamma=0
)

xgb_model.fit(X_train_xgb, y_train_xgb, sample_weight=sample_weights)

y_pred_xgb = xgb_model.predict(X_test_xgb)
print("Accuracy:", accuracy_score(y_test_xgb, y_pred_xgb))
print(classification_report(y_test_xgb, y_pred_xgb, target_names=le.classes_))

y_proba_xgb = xgb_model.predict_proba(X_test_xgb)
roc_auc_xgb = roc_auc_score(y_test_xgb, y_proba_xgb, multi_class='ovr', average='macro')
print("ROC-AUC (macro, ovr):", roc_auc_xgb)


Accuracy: 0.43548387096774194
              precision    recall  f1-score   support

        Down       0.32      0.53      0.40        92
        Flat       0.56      0.45      0.50       265
          Up       0.37      0.35      0.36       139

    accuracy                           0.44       496
   macro avg       0.42      0.44      0.42       496
weighted avg       0.46      0.44      0.44       496

ROC-AUC (macro, ovr): 0.62997714256917


### Multilayer Perceptron (MLP)

In [102]:
from sklearn.neural_network import MLPClassifier

X_mlp = X.dropna()
y_mlp = y.loc[X_mlp.index]

le = LabelEncoder()
y_mlp_enc = le.fit_transform(y_mlp)

split_point = int(len(X_mlp) * 0.8)
X_train_mlp, X_test_mlp = X_mlp.iloc[:split_point], X_mlp.iloc[split_point:]
y_train_mlp, y_test_mlp = y_mlp_enc[:split_point], y_mlp_enc[split_point:]

scaler_mlp = StandardScaler()
X_train_scaled_mlp = scaler_mlp.fit_transform(X_train_mlp)
X_test_scaled_mlp = scaler_mlp.transform(X_test_mlp)

mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42)
mlp_model.fit(X_train_scaled_mlp, y_train_mlp)

y_pred_mlp = mlp_model.predict(X_test_scaled_mlp)
print("Accuracy:", accuracy_score(y_test_mlp, y_pred_mlp))
print(classification_report(y_test_mlp, y_pred_mlp, target_names=le.classes_))

y_proba_mlp = mlp_model.predict_proba(X_test_scaled_mlp)
roc_auc_mlp = roc_auc_score(y_test_mlp, y_proba_mlp, multi_class='ovr', average='macro')
print("ROC-AUC (macro, ovr):", roc_auc_mlp)

Accuracy: 0.5020161290322581
              precision    recall  f1-score   support

        Down       0.35      0.29      0.32        92
        Flat       0.57      0.69      0.62       265
          Up       0.40      0.28      0.33       139

    accuracy                           0.50       496
   macro avg       0.44      0.42      0.42       496
weighted avg       0.48      0.50      0.49       496

ROC-AUC (macro, ovr): 0.5668684469694626


To identify the best approach for predicting the NASDAQ-100’s opening gap direction, a range of machine learning models was tested. The task involves multi-class classification (Up, Down, Flat), based on structured data derived from international market indices. Each model was selected to evaluate how well it could handle time-dependent financial features, class imbalance, and subtle variations in input signals.

Linear Regression (Rounded Output)
We began by testing a linear regression model, modified to classify by rounding predicted outputs. While not designed for classification tasks, it served as a useful baseline to compare against more advanced methods. As expected, the model favored the majority class (“Flat”) and struggled to correctly identify less frequent “Up” and “Down” cases. This confirmed the need for more specialized classification techniques.

Random Forest Classifier
Random Forest was selected for its ability to manage non-linear relationships and its resilience to noise,both common in financial data. It also allows for automatic handling of imbalanced classes through the use of class weights. This model produced balanced results and highlighted the relative importance of input features, helping us understand which global markets contributed most to the prediction.

XGBoost Classifier
We included XGBoost as a more sophisticated tree-based model known for strong performance in classification tasks. It allows for more precise tuning and is effective in situations where small differences in the data carry meaning. To address class imbalance, we applied class-based weighting during training. This improved the model’s ability to recognize “Up” and “Down” cases without overfitting to the “Flat” class. XGBoost showed strong results in terms of class separation and generalization.

Multilayer Perceptron (MLP)
To explore a non-tree-based method, we trained a Multilayer Perceptron neural network. This model is capable of capturing complex relationships among features that may not be easily represented by trees. However, it required careful scaling of input data and tuning of architecture and training duration. The MLP achieved reasonable performance but was more sensitive to data quality and class distribution.

Each model provided a different perspective on the dataset. While simpler models offered fast comparisons, tree-based approaches like Random Forest and XGBoost delivered more reliable classification under noisy and imbalanced conditions. Including a neural network allowed us to test the limits of pattern detection across regional indices.

This comparative analysis gives us a clearer path forward in selecting and refining a model that supports the client’ objective: delivering accurate, actionable opening-gap predictions based on global overnight market behavior.

## Step 5: Model evaluation

In [103]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)
import pandas as pd

# CV object since this is time series data
tscv = TimeSeriesSplit(n_splits=5)

# Plain callable scorer (macro ROC-AUC, OvR)
def macro_ovr_auc(estimator, X, y):
    y_proba = estimator.predict_proba(X)
    return roc_auc_score(y, y_proba, multi_class="ovr", average="macro")

# Baseline - DummyClassifier
print("\n" + "=" * 40 + "\nBaseline: DummyClassifier (most frequent)\n" + "=" * 40)
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train_rf, y_train_rf)
y_dummy_pred = dummy.predict(X_test_rf)
print("Accuracy:", accuracy_score(y_test_rf, y_dummy_pred))
print(
    classification_report(
        y_test_rf, y_dummy_pred, target_names=le.classes_, zero_division=0
    )
)

# Linear Regression (rounded) - test set only
print("\n" + "=" * 40 + "\nLinear Regression (Rounded) - Test Only\n" + "=" * 40)
print("Accuracy:", accuracy_score(y_test_lr, y_pred_class_lr))
print(
    classification_report(
        y_test_lr, y_pred_class_lr, target_names=le.classes_, zero_division=0
    )
)
y_proba_lr = pd.get_dummies(y_pred_class_lr)
roc_auc_lr = roc_auc_score(y_test_lr, y_proba_lr, multi_class="ovr", average="macro")
print("ROC-AUC (macro, ovr):", roc_auc_lr)

# Random Forest - 5-fold TimeSeries CV + test set
print("\n" + "=" * 40 + "\nRandom Forest - 5-Fold TimeSeries CV (ROC-AUC)\n" + "=" * 40)
rf_cv_scores = cross_val_score(
    rf_model, X_rf, y_rf_enc, cv=tscv, scoring=macro_ovr_auc, n_jobs=-1
)
print("CV ROC-AUC Scores:", rf_cv_scores)
print("Mean CV ROC-AUC:", rf_cv_scores.mean())

print("\nRandom Forest - Test Set")
y_pred_rf = rf_model.predict(X_test_rf)
print("Accuracy:", accuracy_score(y_test_rf, y_pred_rf))
print(
    classification_report(
        y_test_rf, y_pred_rf, target_names=le.classes_, zero_division=0
    )
)
y_proba_rf = rf_model.predict_proba(X_test_rf)
print("ROC-AUC (macro, ovr):", roc_auc_score(y_test_rf, y_proba_rf, multi_class="ovr", average="macro"))

# XGBoost - 5-fold TimeSeries CV + test set
print("\n" + "=" * 40 + "\nXGBoost - 5-Fold TimeSeries CV (ROC-AUC)\n" + "=" * 40)
xgb_cv_scores = cross_val_score(
    xgb_model, X_xgb, y_xgb_enc, cv=tscv, scoring=macro_ovr_auc, n_jobs=-1
)
print("CV ROC-AUC Scores:", xgb_cv_scores)
print("Mean CV ROC-AUC:", xgb_cv_scores.mean())

print("\nXGBoost - Test Set")
y_pred_xgb = xgb_model.predict(X_test_xgb)
print("Accuracy:", accuracy_score(y_test_xgb, y_pred_xgb))
print(
    classification_report(
        y_test_xgb, y_pred_xgb, target_names=le.classes_, zero_division=0
    )
)
y_proba_xgb = xgb_model.predict_proba(X_test_xgb)
print("ROC-AUC (macro, ovr):", roc_auc_score(y_test_xgb, y_proba_xgb, multi_class="ovr", average="macro"))

# MLP Classifier - 5-fold TimeSeries CV + test set
mlp_model.set_params(max_iter=1000)

print("\n" + "=" * 40 + "\nMLP Classifier - 5-Fold TimeSeries CV (ROC-AUC)\n" + "=" * 40)
mlp_cv_scores = cross_val_score(
    mlp_model,
    scaler_mlp.transform(X_mlp),
    y_mlp_enc,
    cv=tscv,
    scoring=macro_ovr_auc,
    n_jobs=-1,
)
print("CV ROC-AUC Scores:", mlp_cv_scores)
print("Mean CV ROC-AUC:", mlp_cv_scores.mean())

print("\nMLP Classifier - Test Set")
y_pred_mlp = mlp_model.predict(X_test_scaled_mlp)
print("Accuracy:", accuracy_score(y_test_mlp, y_pred_mlp))
print(
    classification_report(
        y_test_mlp, y_pred_mlp, target_names=le.classes_, zero_division=0
    )
)
y_proba_mlp = mlp_model.predict_proba(X_test_scaled_mlp)
print("ROC-AUC (macro, ovr):", roc_auc_score(y_test_mlp, y_proba_mlp, multi_class="ovr", average="macro"))



Baseline: DummyClassifier (most frequent)
Accuracy: 0.5342741935483871
              precision    recall  f1-score   support

        Down       0.00      0.00      0.00        92
        Flat       0.53      1.00      0.70       265
          Up       0.00      0.00      0.00       139

    accuracy                           0.53       496
   macro avg       0.18      0.33      0.23       496
weighted avg       0.29      0.53      0.37       496


Linear Regression (Rounded) - Test Only
Accuracy: 0.530241935483871
              precision    recall  f1-score   support

        Down       0.57      0.04      0.08        92
        Flat       0.54      0.95      0.68       265
          Up       0.39      0.05      0.09       139

    accuracy                           0.53       496
   macro avg       0.50      0.35      0.28       496
weighted avg       0.50      0.53      0.41       496

ROC-AUC (macro, ovr): 0.5097485589616698

Random Forest - 5-Fold TimeSeries CV (ROC-AUC)
CV ROC-A

## Step 6: Hyperparameter tuning

In [104]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV
from collections import Counter
import pandas as pd

# Preprocess data
X_xgb = X.dropna()
y_xgb = y.loc[X_xgb.index]

le = LabelEncoder()
y_xgb_enc = pd.Series(le.fit_transform(y_xgb), index=y_xgb.index)

split_point = int(len(X_xgb) * 0.8)
X_train, X_test = X_xgb.iloc[:split_point], X_xgb.iloc[split_point:]
y_train, y_test = y_xgb_enc.iloc[:split_point], y_xgb_enc.iloc[split_point:]

# Handle class imbalance
counts = Counter(y_train)
weights = {cls: max(counts.values()) / cnt for cls, cnt in counts.items()}
sample_weights = y_train.map(weights)

# Parameter grid for hyperparameter tuning
param_grid = {
    "max_depth": [3, 5, 7],
    "booster": ["gbtree", "dart"],
    "learning_rate": [0.005, 0.01, 0.05],
    "n_estimators": [100, 200],
    "min_child_weight": [3, 7, 9],
    "gamma": [0, 0.3],
    "reg_alpha": [0.3, 0.5],
    "reg_lambda": [0.5, 1]
}

# Define base model
xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss",
    verbosity=0
)

# Run GridSearchCV
search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=3,
    verbose=2,
    n_jobs=-1
)

search.fit(X_train, y_train, sample_weight=sample_weights)

# Best model
best_model = search.best_estimator_

# Evaluate on test set
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ROC-AUC score
y_proba = best_model.predict_proba(X_test)
roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
print("ROC-AUC (macro, ovr):", roc_auc)

# Output best params
print("Best Parameters:", search.best_params_)


Fitting 3 folds for each of 864 candidates, totalling 2592 fits
Accuracy: 0.47580645161290325
              precision    recall  f1-score   support

        Down       0.34      0.59      0.43        92
        Flat       0.61      0.50      0.55       265
          Up       0.41      0.36      0.38       139

    accuracy                           0.48       496
   macro avg       0.45      0.48      0.45       496
weighted avg       0.51      0.48      0.48       496

ROC-AUC (macro, ovr): 0.6342450443230844
Best Parameters: {'booster': 'gbtree', 'gamma': 0.3, 'learning_rate': 0.005, 'max_depth': 3, 'min_child_weight': 9, 'n_estimators': 100, 'reg_alpha': 0.5, 'reg_lambda': 1}


In [105]:
import joblib
joblib.dump(best_model, "xgb_best_model.pkl")
joblib.dump(le, "best_model.pkl")

['best_model.pkl']

Validation scheme
- TimeSeriesSplit (3 folds) keeps chronological order which is essential
  for stock data; prevents look-ahead bias present in random K-folds.

 Metric choice
- Macro ROC-AUC (OvR) is preferred over accuracy, because:
    1. Class imbalance (Flat dominates) makes accuracy misleading.
    2. ROC-AUC measures quality of the probability ranking across classes.

 Search strategy
    - GridSearchCV, not random search, because the grid (768 combos)
  is small enough to exhaustively test and ensures every key setting
  combination is evaluated.

Hyperparameters tuned and why

 - max_depth    -    controls tree complexity; smaller depth reduces over-fit.
 - booster   -   gbtree vs dart; dart can drop trees during boosting and
                  sometimes generalises better.
 - learning_rate  -  lower rates require more trees but improve stability.
 - n_estimators  -   paired with learning_rate to balance bias vs variance.
 - min_child_weight - higher values force larger leaf samples, adding regularisation.
 - gamma    -    minimum loss reduction to split; combats unnecessary splits.
 - reg_alpha   -   L1 penalty; useful for sparse gain pruning.
 - reg_lambda   -    L2 penalty; reduces leaf weight magnitudes.

 Evidence of improvement
 - Baseline XGB (default)  macro ROC-AUC 0.629   accuracy 0.435
 - Tuned XGB (grid best)   macro ROC-AUC 0.634   accuracy 0.476
 - The tuned model raises AUC by 0.005 and improves the recall of the minority classes.

 Final model chosen for deployment:

 - booster          = "gbtree"
 - max_depth        = 3
 - learning_rate    = 0.005
 - n_estimators     = 100
 - min_child_weight = 9
 - gamma            = 0.3
 - reg_alpha        = 0.5
 - reg_lambda       = 1

 This configuration achieved the highest cross-validated macro ROC-AUC
 and the most balanced precision-recall on the hold-out set, so it will be
 retained for production use.


## Step 7: Testing and backtesting

In [106]:
# Perform final testing or backtesting
# Discuss final model strengths and limitations
import joblib

final_model = joblib.load('xgb_best_model.pkl')
y_pred = final_model.predict(X_test)

print("Final Model Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC (macro):", roc_auc_score(y_test, final_model.predict_proba(X_test), multi_class='ovr', average='macro'))

Final Model Accuracy: 0.47580645161290325
              precision    recall  f1-score   support

           0       0.34      0.59      0.43        92
           1       0.61      0.50      0.55       265
           2       0.41      0.36      0.38       139

    accuracy                           0.48       496
   macro avg       0.45      0.48      0.45       496
weighted avg       0.51      0.48      0.48       496

ROC-AUC (macro): 0.6342450443230844


## Step 8: Save the final results

In [107]:
results_df = X_test.copy()
results_df["Predicted_Direction"] = final_model.predict(X_test)
results_df["True_Direction"] = y_test
results_df.to_csv("final_results_table.csv")


## Step 9: Preparation for deployment

In [108]:
# Save model(s) for deployment
import joblib
joblib.dump(best_model, "xgb_best_model.pkl")
joblib.dump(le, "best_model.pkl")

['best_model.pkl']

## Step 10: Export the features and labels for the dashboard

In [ ]:
X.to_csv("Features.csv")
y.to_csv("Target.csv")